In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from load import load_data, tensorize_data
from models import NaiveRNN, NaiveLSTM

In [ ]:
def prepareDataLoader(dataset, labels, batchsize=32):
    train, test, train_labels, test_labels = train_test_split(dataset, labels, test_size=0.2, random_state=42)


    train_dataset = TensorDataset(train, train_labels)
    test_dataset = TensorDataset(test, test_labels)

    train_dataloader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

    return (train_dataloader, test_dataloader)

In [ ]:
def training(epochs, model, dataloader, criterion, optimizer):
    for epoch in range(epochs):
        model.train()
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

def evaluate(model, dataloader):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Accuracy: {100 * correct / total}%')


In [4]:
dfs = load_data()
con4_df = dfs[15][0]
con5_df = dfs[17][0]

assert(dfs[15][1] == "PFC_con_4.csv")
assert(dfs[17][1] == "PFC_con_5.csv")

In [ ]:
# Use DS+/DS- as labels (0)
tensors_4 = tensorize_data(con4_df, 0)
labels = tensors_4[1]
dataset = tensors_4[0]

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rnn_model = NaiveRNN(1, 32, 2).to(device)
lstm_model = NaiveLSTM(1, 32, 2).to(device)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)



training(20, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.6965
Epoch [2/20], Loss: 0.6879
Epoch [3/20], Loss: 0.7111
Epoch [4/20], Loss: 0.6859
Epoch [5/20], Loss: 0.6771
Epoch [6/20], Loss: 0.7159
Epoch [7/20], Loss: 0.6752
Epoch [8/20], Loss: 0.6840
Epoch [9/20], Loss: 0.6793
Epoch [10/20], Loss: 0.6994
Epoch [11/20], Loss: 0.7016
Epoch [12/20], Loss: 0.6617
Epoch [13/20], Loss: 0.6825
Epoch [14/20], Loss: 0.6859
Epoch [15/20], Loss: 0.6727
Epoch [16/20], Loss: 0.7058
Epoch [17/20], Loss: 0.7332
Epoch [18/20], Loss: 0.7036
Epoch [19/20], Loss: 0.6879
Epoch [20/20], Loss: 0.7055


ValueError: too many values to unpack (expected 2)